# Step 1b — Per-animal distributions of each spike metric

| | |
|---|---|
| **Original** | `Liam_work/tests/non_behavioral_spike_analysis/Visualizations_features.ipynb` |
| **Status** | QC / exploration. Reads step-1 output. |

For every metric in the clean spike-metric tables, plots each neuron's value grouped by animal (sorted by
age, coloured by genotype) as scatter + box and as per-animal KDEs. This is where animal-level confounds
were spotted, e.g. `min_isis` sitting at the hardware sampling floor (0.001 s or 0.00056 s depending on
rig) and `caf104`'s extreme long ISIs — the reasons `min_isis` is dropped and `caf104` is excluded in the
classifiers.

Note: unlike the classifier notebooks, this one sets `EXCLUDE_ANIMALS = []` and `DROP_FEATURES = []` so you
see everything.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import pickle
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 8)

RESULTS_DIR = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/spike_metrics/'  # repo copy (unused below; kept from original)

AGE_CUTOFF_DAYS = 200  # young vs old boundary

print('Imports loaded.')

In [ ]:
# =============================================================================
# LOAD PRE-COMPUTED CLEAN SPIKE METRICS
# =============================================================================
# These files are created by compute_clean_spikemetrics.ipynb (run it first).
# They contain metrics recomputed from raw ISIs with negative values filtered out.

# Repo copy: read the clean metric tables from the project's data/ folder.
DATA_DIR = '/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/data/spike_metrics/'

META_COLS = ['animal', 'age', 'sex', 'geno', 'brain_region', 'cell_type']

# ---- EXCLUSION CONFIG (edit these to add/remove) ----

# Animals to exclude:
#   caf104 — spike sorting instability. 81% of its neurons have within-epoch
#   ISIs > 100s (next closest animal: 9.7%). This inflates max_isis, sd, cv,
#   and dist_alpha, making caf104 a massive outlier that dominates the classifier.
EXCLUDE_ANIMALS = []

# Features to drop:
#   min_isis  — measures recording sampling rate, not biology. ~70% of neurons
#               sit at exactly the hardware floor (0.001s or 0.00056s).
#   mean_ifr  — exact inverse of mean_isis (correlation = -1.0), adds nothing.
DROP_FEATURES = []

# All metrics from the pkl files (before dropping)
ALL_METRICS = ['mean_isis', 'mean_ifr', 'median_isis', 'max_isis', 'min_isis',
               'sd', 'cv', 'cv2', 'lv', 'lvr', 'dist_alpha', 'dist_beta']

# Metrics that actually get used (after dropping)
METRIC_NAMES = [m for m in ALL_METRICS if m not in DROP_FEATURES]

# ---- LOAD ----

state_dfs = {}
for state in ['nrem', 'wake', 'rem']:
    path = os.path.join(DATA_DIR, f'clean_spikemetrics_{state}.pkl')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{path} not found. Run compute_clean_spikemetrics.ipynb first!')
    state_dfs[state] = pd.read_pickle(path)
    print(f'Loaded {state}: {state_dfs[state].shape}')

# ---- EXCLUDE ANIMALS ----

if EXCLUDE_ANIMALS:
    for state in state_dfs:
        before = len(state_dfs[state])
        state_dfs[state] = state_dfs[state][~state_dfs[state]['animal'].isin(EXCLUDE_ANIMALS)]
        state_dfs[state] = state_dfs[state].reset_index(drop=True)
    after = len(state_dfs['nrem'])
    print(f'\nExcluded animals {EXCLUDE_ANIMALS}: {before} -> {after} neurons')

if DROP_FEATURES:
    print(f'Dropped features: {DROP_FEATURES}')
    print(f'Remaining features per state: {len(METRIC_NAMES)} ({METRIC_NAMES})')

# ---- VERIFY ALIGNMENT ----

assert len(state_dfs['nrem']) == len(state_dfs['wake']) == len(state_dfs['rem']), \
    'State DataFrames have different lengths!'
assert (state_dfs['nrem']['animal'].values == state_dfs['wake']['animal'].values).all(), \
    'Animal order mismatch between nrem and wake!'

# ---- BUILD UNIFIED DATAFRAME ----

df = state_dfs['nrem'][META_COLS].copy()

STATE_FEATURES = {}
for state in ['nrem', 'wake', 'rem']:
    state_cols = []
    for col in METRIC_NAMES:
        new_col = f'{state}_{col}'
        df[new_col] = state_dfs[state][col].values
        state_cols.append(new_col)
    STATE_FEATURES[state] = state_cols

COMBINED_COLS = []
for state in ['nrem', 'wake', 'rem']:
    COMBINED_COLS.extend(STATE_FEATURES[state])
STATE_FEATURES['combined'] = COMBINED_COLS

print(f'\nFeature sets:')
for state, cols in STATE_FEATURES.items():
    print(f'  {state:10s}: {len(cols)} features')

# ---- LABELS ----

df['genotype'] = df['geno'].map(lambda x: 'AD' if 'app' in str(x).lower() else 'WT')
df['age_group'] = df['age'].apply(lambda x: 'young' if x < AGE_CUTOFF_DAYS else 'old')
df['group'] = df['age_group'] + '_' + df['genotype']

# ---- CLEAN NaN/inf ----

n_before = len(df)
df = df.dropna(subset=COMBINED_COLS)
print(f'\nDropped {n_before - len(df)} rows with NaN metrics ({len(df)} remaining)')

mask = np.isfinite(df[COMBINED_COLS].values).all(axis=1)
df = df[mask].copy()
print(f'After removing inf: {len(df)} neurons')

# ---- SUMMARY ----

print(f'\n{"="*60}')
print(f'DATASET SUMMARY')
print(f'{"="*60}')
print(f'Total neurons: {len(df)}')
print(f'Excluded animals: {EXCLUDE_ANIMALS}')
print(f'Dropped features: {DROP_FEATURES}')
print(f'\nBy group:')
for group in sorted(df['group'].unique()):
    sub = df[df['group'] == group]
    animals = sorted(sub['animal'].unique())
    print(f'  {group:12s}: {len(sub):4d} neurons, {len(animals)} animals: {animals}')

print(f'\nAge cutoff: {AGE_CUTOFF_DAYS} days')
print(f'\nAnimal ages:')
for animal in sorted(df['animal'].unique()):
    sub = df[df['animal'] == animal]
    print(f'  {animal:8s}: age={sub["age"].iloc[0]:7.1f}d  geno={sub["genotype"].iloc[0]:2s}  '
          f'group={sub["group"].iloc[0]:10s}  n={len(sub)}')

In [ ]:
# =============================================================================
# PER-ANIMAL FEATURE DISTRIBUTIONS — SCATTER + BOXPLOT
# =============================================================================
# For each feature, plot every neuron's value grouped by animal, colored by genotype.
# This reveals animal-level confounds (e.g. sampling rate artifacts in min_isis).

# Sort animals by age for consistent x-axis ordering
animal_order = (
    df.groupby('animal').agg(age=('age', 'first'), geno=('genotype', 'first'))
    .sort_values('age')
)
animal_list = animal_order.index.tolist()
geno_colors = {'WT': '#1565C0', 'AD': '#C62828'}
animal_colors = [geno_colors[animal_order.loc[a, 'geno']] for a in animal_list]

for state in ['nrem', 'wake', 'rem']:
    for i, metric in enumerate(METRIC_NAMES):
        fig, ax = plt.subplots(figsize=(18, 6))
        col = f'{state}_{metric}'
        vals = df[col].values

        # Clip to 1st-99th percentile for visualization
        p1, p99 = np.nanpercentile(vals, [1, 99])

        for j, animal in enumerate(animal_list):
            mask = df['animal'] == animal
            y = df.loc[mask, col].values
            y_clip = np.clip(y, p1, p99)
            x = np.random.default_rng(42).normal(j, 0.15, size=len(y_clip))
            color = geno_colors[animal_order.loc[animal, 'geno']]
            ax.scatter(x, y_clip, s=4, alpha=0.35, color=color, rasterized=True)

        # Boxplot overlay
        bp_data = [np.clip(df.loc[df['animal'] == a, col].dropna().values, p1, p99)
                    for a in animal_list]
        bp = ax.boxplot(bp_data, positions=range(len(animal_list)),
                        widths=0.5, showfliers=False, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 1.5},
                        whiskerprops={'linewidth': 0.8},
                        boxprops={'linewidth': 0.5})
        for patch, color in zip(bp['boxes'], animal_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.2)

        ax.set_xticks(range(len(animal_list)))
        ax.set_xticklabels(animal_list, rotation=45, fontsize=10, ha='right')
        ax.set_ylabel(metric, fontsize=12)
        ax.set_title(f'{state.upper()} — {metric}', fontsize=14, fontweight='bold')
        ax.tick_params(axis='y', labelsize=10)

        # Legend
        from matplotlib.lines import Line2D
        legend_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
                                 markersize=10, label=g) for g, c in geno_colors.items()]
        ax.legend(handles=legend_handles, fontsize=11, loc='upper right')
        plt.tight_layout()
        plt.show()

In [ ]:
# =============================================================================
# PER-ANIMAL FEATURE DISTRIBUTIONS — KDE CURVES
# =============================================================================
# One subplot per animal, arranged in a grid. Each animal gets its own panel
# so curves don't overlap and you can clearly see each distribution.
# Animals sorted by age (left to right, top to bottom). AD = red, WT = blue.
# A shared gray "all animals" reference curve is shown in each panel.

from scipy.stats import gaussian_kde

n_animals = len(animal_list)
ncols = 5
nrows = int(np.ceil(n_animals / ncols))

for state in ['nrem', 'wake', 'rem']:
    for metric in METRIC_NAMES:
        col = f'{state}_{metric}'
        all_vals = df[col].dropna().values
        p1, p99 = np.nanpercentile(all_vals, [1, 99])
        x_grid = np.linspace(p1, p99, 200)

        # Precompute global KDE for reference
        all_clipped = all_vals[(all_vals >= p1) & (all_vals <= p99)]
        try:
            global_kde = gaussian_kde(all_clipped)
            global_density = global_kde(x_grid)
            global_density = global_density / global_density.max()
        except:
            global_density = None

        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows),
                                 sharex=True, sharey=True)
        axes_flat = axes.flat if hasattr(axes, 'flat') else [axes]

        for j, animal in enumerate(animal_list):
            ax = axes_flat[j]

            # Reference curve (all animals pooled)
            if global_density is not None:
                ax.fill_between(x_grid, global_density, alpha=0.12, color='gray')

            # This animal's KDE
            vals = df.loc[df['animal'] == animal, col].dropna().values
            vals = vals[(vals >= p1) & (vals <= p99)]
            if len(vals) >= 5:
                try:
                    kde = gaussian_kde(vals)
                    density = kde(x_grid)
                    density = density / density.max()
                    geno = animal_order.loc[animal, 'geno']
                    color = geno_colors[geno]
                    ax.plot(x_grid, density, color=color, linewidth=2)
                    ax.fill_between(x_grid, density, alpha=0.25, color=color)
                except np.linalg.LinAlgError:
                    pass

            age = animal_order.loc[animal, 'age']
            geno = animal_order.loc[animal, 'geno']
            n_neurons = (df['animal'] == animal).sum()
            ax.set_title(f'{animal}  ({geno}, {age:.0f}d, n={n_neurons})',
                         fontsize=10, fontweight='bold',
                         color=geno_colors[geno])
            ax.set_yticks([])
            if j >= (nrows - 1) * ncols:
                ax.set_xlabel(metric, fontsize=9)

        # Hide unused subplots
        for j in range(n_animals, nrows * ncols):
            axes_flat[j].set_visible(False)

        fig.suptitle(f'{state.upper()} — {metric}', fontsize=15, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()